# Open Door Legal — Client Feedback Analysis

Analysis of 324 client feedback responses covering:
- **Net Promoter Score & quantitative ratings**
- **Topic modeling** — what clients say ODL does well vs. what could improve
- **Sentiment analysis** — tone of open-ended responses
- **Breakdowns** by case owner and degree of resolution

## 0. Setup

In [ ]:
# !pip install vaderSentiment pandas numpy matplotlib seaborn scikit-learn nltk openpyxl
# import nltk; nltk.download('stopwords'); nltk.download('vader_lexicon')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

analyzer = SentimentIntensityAnalyzer()
STOP_WORDS = set(stopwords.words('english'))

# Domain-specific stop words to remove from topic models
ODL_STOPS = {'open', 'door', 'legal', 'odl', 'help', 'helped', 'help',
             'would', 'could', 'also', 'one', 'really', 'think', 'feel',
             'know', 'get', 'got', 'make', 'made', 'like', 'need', 'needed'}
ALL_STOPS = STOP_WORDS | ODL_STOPS

## 1. Load Data

In [ ]:
df = pd.read_excel('data/report_feedback_filtered.xlsx')

# Rename columns to shorter handles
df = df.rename(columns={
    'Client Feedback: Created Date':        'date',
    'Case: Client Name':                    'client_name',
    'Case: Case Number':                    'case_number',
    'Case: Subject':                        'case_subject',
    'Case: Degree of Resolution':           'resolution',
    'Case: Case Owner':                     'case_owner',
    'Net Promoter Score':                   'nps',
    'How much of a positive diff has ODL had': 'positive_diff',
    'What does Open Door Legal do well':    'do_well',
    'What could Open Door Legal do better': 'do_better',
    'How well has ODL met your needs':      'needs_met',
    'Informed About Case':                  'informed',
    'Did we give choices':                  'gave_choices',
    'Barriers to services':                 'barriers',
    'Anything we could not have helped with?': 'beyond_scope',
})

df['date'] = pd.to_datetime(df['date'], errors='coerce')

print(f'Loaded {len(df):,} responses')
print(f'Date range: {df["date"].min().date()} to {df["date"].max().date()}')
df.head(3)

## 2. Quantitative Overview

In [ ]:
# NPS categories
def nps_category(score):
    if score >= 9:  return 'Promoter'
    if score >= 7:  return 'Passive'
    return 'Detractor'

df['nps_category'] = df['nps'].apply(nps_category)

nps_score = (
    (df['nps_category'] == 'Promoter').mean() -
    (df['nps_category'] == 'Detractor').mean()
) * 100

print(f'Net Promoter Score: {nps_score:.1f}')
print(df['nps_category'].value_counts().to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# NPS score distribution
sns.histplot(df['nps'], bins=10, discrete=True, ax=axes[0],
             color='steelblue', edgecolor='white')
axes[0].set_title('Net Promoter Score Distribution')
axes[0].set_xlabel('Score (0–10)')
axes[0].set_ylabel('Number of clients')
for x, color in [(range(0,7),'tomato'), (range(7,9),'gold'), (range(9,11),'mediumseagreen')]:
    for xi in x:
        axes[0].axvspan(xi-0.5, xi+0.5, alpha=0.08, color=color)

# NPS category breakdown
cat_counts = df['nps_category'].value_counts()
colors = {'Promoter':'mediumseagreen','Passive':'gold','Detractor':'tomato'}
bars = axes[1].bar(cat_counts.index, cat_counts.values,
                   color=[colors[c] for c in cat_counts.index])
axes[1].bar_label(bars, fmt='%d', padding=3)
axes[1].set_title(f'NPS Categories  (NPS = {nps_score:.0f})')
axes[1].set_ylabel('Count')

plt.suptitle('Net Promoter Score', fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Positive difference
pos_diff = df['positive_diff'].value_counts()
order = ['Extreme','High','Moderate','Low','None']
order = [o for o in order if o in pos_diff.index]
pd_colors = {'Extreme':'#2ecc71','High':'#82e0aa','Moderate':'#f4d03f',
             'Low':'#e59866','None':'#e74c3c'}
axes[0].bar(order, [pos_diff.get(o, 0) for o in order],
            color=[pd_colors.get(o,'steelblue') for o in order])
axes[0].set_title('How much of a positive difference\nhas ODL had?')
axes[0].set_ylabel('Count')
for i, o in enumerate(order):
    v = pos_diff.get(o, 0)
    axes[0].text(i, v + 1, str(v), ha='center', fontsize=9)

# Degree of resolution
res = df['resolution'].value_counts().dropna()
res_colors = {'Positive':'mediumseagreen','Neutral':'steelblue','Negative':'tomato'}
axes[1].bar(res.index, res.values,
            color=[res_colors.get(r,'grey') for r in res.index])
axes[1].set_title('Degree of Resolution')
axes[1].set_ylabel('Count')
for i, (idx, v) in enumerate(res.items()):
    axes[1].text(i, v + 1, str(v), ha='center', fontsize=9)

plt.suptitle('Client Satisfaction Ratings', fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# Yes/No questions
questions = {
    'Informed about case': df['informed'],
    'Gave choices':        df['gave_choices'],
    'Barriers to service': df['barriers'],
    'Beyond scope':        df['beyond_scope'],
}

fig, axes = plt.subplots(1, len(questions), figsize=(14, 4))
for ax, (label, col) in zip(axes, questions.items()):
    counts = col.value_counts()
    colors = ['mediumseagreen' if i == 'Yes' else 'tomato' if i == 'No' else 'steelblue'
              for i in counts.index]
    bars = ax.bar(counts.index, counts.values, color=colors)
    ax.bar_label(bars, fmt='%d', padding=2)
    total = counts.sum()
    pct = (counts / total * 100).round(1)
    ax.set_title(f'{label}\n({pct.get("Yes",0):.0f}% Yes)', fontsize=10)
    ax.set_ylabel('Count' if ax == axes[0] else '')

plt.suptitle('Yes / No Questions', fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# NPS by case owner (owners with 5+ responses)
owner_counts = df['case_owner'].value_counts()
active_owners = owner_counts[owner_counts >= 5].index

owner_nps = (
    df[df['case_owner'].isin(active_owners)]
    .groupby('case_owner')['nps']
    .agg(['mean','count'])
    .round(2)
    .sort_values('mean', ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(owner_nps.index, owner_nps['mean'], color='steelblue')
ax.axvline(df['nps'].mean(), color='tomato', linestyle='--', label=f'Overall mean ({df["nps"].mean():.1f})')
for i, (idx, row) in enumerate(owner_nps.iterrows()):
    ax.text(row['mean'] + 0.05, i, f"{row['mean']:.1f} (n={int(row['count'])})", va='center', fontsize=8)
ax.set_title('Average NPS by Case Owner (≥5 responses)')
ax.set_xlabel('Mean NPS')
ax.set_xlim(0, 11)
ax.legend()
plt.tight_layout(); plt.show()

## 3. Text Preprocessing

In [ ]:
def clean_text(text) -> str:
    if pd.isna(text) or str(text).strip() in ('', 'nan', 'N/A', 'n/a', 'No', 'None'):
        return ''
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = [t for t in text.split() if t not in ALL_STOPS and len(t) > 2]
    return ' '.join(tokens)

df['do_well_clean']   = df['do_well'].apply(clean_text)
df['do_better_clean'] = df['do_better'].apply(clean_text)

# Working subsets (non-empty responses)
dw = df[df['do_well_clean'].str.len() > 0].copy()
db = df[df['do_better_clean'].str.len() > 0].copy()

print(f'"Do well" responses with text:   {len(dw)}')
print(f'"Do better" responses with text: {len(db)}')

## 4. VADER Sentiment Analysis

In [ ]:
def vader_scores(text) -> dict:
    if not text or pd.isna(text):
        return {'compound': 0, 'pos': 0, 'neu': 0, 'neg': 0, 'label': 'neutral'}
    scores = analyzer.polarity_scores(str(text))
    scores['label'] = (
        'positive' if scores['compound'] >= 0.05
        else 'negative' if scores['compound'] <= -0.05
        else 'neutral'
    )
    return scores

# Score both columns on raw (not cleaned) text to preserve sentiment cues
for col, prefix in [('do_well', 'dw'), ('do_better', 'db')]:
    scores = df[col].apply(vader_scores).apply(pd.Series)
    scores.columns = [f'{prefix}_{c}' for c in scores.columns]
    df = pd.concat([df, scores], axis=1)

print('"Do well" sentiment:')
print(df['dw_label'].value_counts().to_string())
print()
print('"Do better" sentiment:')
print(df['db_label'].value_counts().to_string())

In [ ]:
palette = {'positive':'mediumseagreen','neutral':'steelblue','negative':'tomato'}
order   = ['positive','neutral','negative']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, prefix, title in [
    (axes[0], 'dw', 'What ODL does well'),
    (axes[1], 'db', 'What could be better'),
]:
    counts = df[f'{prefix}_label'].value_counts()
    pcts   = (counts / counts.sum() * 100).round(1)
    bars = ax.bar(
        [l.capitalize() for l in order],
        [counts.get(l, 0) for l in order],
        color=[palette[l] for l in order]
    )
    for bar, l in zip(bars, order):
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 1,
                f'{h}\n({pcts.get(l,0):.0f}%)', ha='center', fontsize=9)
    ax.set_title(title)
    ax.set_ylabel('Number of responses')
    ax.set_ylim(0, counts.max() * 1.2)

plt.suptitle('Sentiment of Open-Ended Responses', fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# Sentiment compound score by degree of resolution
res_sent = (
    df.dropna(subset=['resolution'])
    .groupby('resolution')[['dw_compound','db_compound']]
    .mean().round(3)
)
res_sent.columns = ['Do well (mean compound)', 'Do better (mean compound)']
print('Mean VADER compound score by resolution:')
display(res_sent)

fig, ax = plt.subplots(figsize=(8, 4))
res_sent.plot.bar(ax=ax, color=['mediumseagreen','steelblue'], edgecolor='white')
ax.axhline(0, color='grey', linewidth=0.8)
ax.set_title('Mean Sentiment Score by Degree of Resolution')
ax.set_xlabel('')
ax.set_ylabel('Mean VADER Compound Score')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(loc='lower right')
plt.tight_layout(); plt.show()

## 5. Topic Modeling — What ODL Does Well

In [ ]:
N_TOPICS_WELL = 5
N_TOP_WORDS   = 8

count_vec_dw = CountVectorizer(max_features=3000, min_df=2, ngram_range=(1,2))
count_mat_dw = count_vec_dw.fit_transform(dw['do_well_clean'])
vocab_dw     = count_vec_dw.get_feature_names_out()

lda_dw = LatentDirichletAllocation(
    n_components=N_TOPICS_WELL, random_state=42,
    learning_method='batch', max_iter=30
)
lda_dw.fit(count_mat_dw)

doc_topics_dw = lda_dw.transform(count_mat_dw)
dw['topic']        = doc_topics_dw.argmax(axis=1)
dw['topic_weight'] = doc_topics_dw.max(axis=1)

def top_words_table(model, vocab, n=8):
    rows = []
    for i, comp in enumerate(model.components_):
        words = [vocab[j] for j in comp.argsort()[:-n-1:-1]]
        rows.append({'Topic': f'Topic {i}', 'Top Words': ', '.join(words)})
    return pd.DataFrame(rows)

print('Topics — What ODL does well:')
display(top_words_table(lda_dw, vocab_dw, N_TOP_WORDS))

In [ ]:
# ── Label topics after reviewing the top words above ──────────────────────
# Edit these to match what you see in the topic output
TOPIC_LABELS_WELL = {
    0: 'Topic 0',
    1: 'Topic 1',
    2: 'Topic 2',
    3: 'Topic 3',
    4: 'Topic 4',
}
dw['topic_label'] = dw['topic'].map(TOPIC_LABELS_WELL)

In [ ]:
topic_counts_dw = dw['topic_label'].value_counts().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(topic_counts_dw.index, topic_counts_dw.values, color='mediumseagreen')
ax.bar_label(bars, fmt='%d', padding=3)
ax.set_title('What ODL Does Well — Topic Prevalence')
ax.set_xlabel('Number of responses')
plt.tight_layout(); plt.show()

## 6. Topic Modeling — What Could Be Better

In [ ]:
N_TOPICS_BETTER = 5

count_vec_db = CountVectorizer(max_features=3000, min_df=2, ngram_range=(1,2))
count_mat_db = count_vec_db.fit_transform(db['do_better_clean'])
vocab_db     = count_vec_db.get_feature_names_out()

lda_db = LatentDirichletAllocation(
    n_components=N_TOPICS_BETTER, random_state=42,
    learning_method='batch', max_iter=30
)
lda_db.fit(count_mat_db)

doc_topics_db = lda_db.transform(count_mat_db)
db['topic']        = doc_topics_db.argmax(axis=1)
db['topic_weight'] = doc_topics_db.max(axis=1)

print('Topics — What could be better:')
display(top_words_table(lda_db, vocab_db, N_TOP_WORDS))

In [ ]:
# ── Label topics after reviewing the top words above ──────────────────────
TOPIC_LABELS_BETTER = {
    0: 'Topic 0',
    1: 'Topic 1',
    2: 'Topic 2',
    3: 'Topic 3',
    4: 'Topic 4',
}
db['topic_label'] = db['topic'].map(TOPIC_LABELS_BETTER)

In [ ]:
topic_counts_db = db['topic_label'].value_counts().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(topic_counts_db.index, topic_counts_db.values, color='steelblue')
ax.bar_label(bars, fmt='%d', padding=3)
ax.set_title('What Could Be Better — Topic Prevalence')
ax.set_xlabel('Number of responses')
plt.tight_layout(); plt.show()

## 7. Topic + Sentiment Combo (Key Finding Chart)

In [ ]:
# Sentiment breakdown per topic — Do well
ct_dw = pd.crosstab(dw['topic_label'], dw['dw_label'], normalize='index').round(3) * 100
for col in ['positive','neutral','negative']:
    if col not in ct_dw.columns:
        ct_dw[col] = 0
ct_dw = ct_dw[['negative','neutral','positive']]

fig, ax = plt.subplots(figsize=(10, 5))
ct_dw.plot.bar(stacked=True, ax=ax,
               color=['tomato','steelblue','mediumseagreen'],
               edgecolor='white')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title('Sentiment by Topic — What ODL Does Well')
ax.set_xlabel('')
ax.set_ylabel('% of responses')
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha='right')
ax.legend(title='Sentiment', bbox_to_anchor=(1.01, 1))
plt.tight_layout(); plt.show()

In [ ]:
# Sentiment breakdown per topic — Do better
ct_db = pd.crosstab(db['topic_label'], db['db_label'], normalize='index').round(3) * 100
for col in ['positive','neutral','negative']:
    if col not in ct_db.columns:
        ct_db[col] = 0
ct_db = ct_db[['negative','neutral','positive']]

fig, ax = plt.subplots(figsize=(10, 5))
ct_db.plot.bar(stacked=True, ax=ax,
               color=['tomato','steelblue','mediumseagreen'],
               edgecolor='white')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title('Sentiment by Topic — What Could Be Better')
ax.set_xlabel('')
ax.set_ylabel('% of responses')
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha='right')
ax.legend(title='Sentiment', bbox_to_anchor=(1.01, 1))
plt.tight_layout(); plt.show()

## 8. Trends Over Time

In [ ]:
monthly = (
    df.set_index('date')
    .resample('M')
    .agg(
        nps_mean=('nps','mean'),
        dw_sentiment=('dw_compound','mean'),
        response_count=('nps','count')
    )
    .dropna()
)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

axes[0].plot(monthly.index, monthly['nps_mean'], marker='o', color='steelblue')
axes[0].axhline(monthly['nps_mean'].mean(), color='grey', linestyle='--', linewidth=0.8)
axes[0].set_ylabel('Mean NPS')
axes[0].set_title('Monthly Trends')
axes[0].set_ylim(0, 10)

axes[1].plot(monthly.index, monthly['dw_sentiment'], marker='o', color='mediumseagreen')
axes[1].axhline(0, color='grey', linestyle='--', linewidth=0.8)
axes[1].fill_between(monthly.index, monthly['dw_sentiment'], 0,
                     where=monthly['dw_sentiment'] >= 0, alpha=0.2, color='mediumseagreen')
axes[1].fill_between(monthly.index, monthly['dw_sentiment'], 0,
                     where=monthly['dw_sentiment'] < 0,  alpha=0.2, color='tomato')
axes[1].set_ylabel('Mean Sentiment\n("Do well" responses)')

plt.tight_layout(); plt.show()

## 9. Export Results

In [ ]:
export_cols = [
    'date','client_name','case_number','case_owner','resolution',
    'nps','nps_category','positive_diff',
    'do_well','dw_compound','dw_label',
    'do_better','db_compound','db_label',
    'informed','gave_choices','barriers'
]
export_cols = [c for c in export_cols if c in df.columns]
df[export_cols].to_csv('odl_feedback_results.csv', index=False)
print('Saved → odl_feedback_results.csv')

top_words_table(lda_dw, vocab_dw, N_TOP_WORDS).to_csv('topics_do_well.csv', index=False)
top_words_table(lda_db, vocab_db, N_TOP_WORDS).to_csv('topics_do_better.csv', index=False)
print('Saved → topics_do_well.csv')
print('Saved → topics_do_better.csv')